In [2]:
"""
==============================================================================
COPYRIGHT & INTELLECTUAL PROPERTY NOTICE
Copyright (c) 2026 Eduardo Ayala Tovar
Title: EXP07 — Beatriz Fire Test (Dense Neural Vector Gate & Offline Benchmark)
License: PolyForm Noncommercial License 1.0.0
==============================================================================
"""

import os
import gc
import json
import math
import time
import random
import hashlib
from enum import Enum
from typing import Dict, Any, List

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==============================================================================
# CONFIGURACIÓN DEL EXPERIMENTO (EXP07 - 100% OFFLINE)
# ==============================================================================
AUTHOR = "Eduardo Ayala Tovar"
LICENSE = "PolyForm Noncommercial License 1.0.0"
EXPERIMENT = "EXP07 — Beatriz Fire Test (Dense Neural Vector Gate & Offline Benchmark)"
YEAR = "2026"

SEEDS = [11, 22, 33]
EPOCHS = 8
DRAWS_PER_EPOCH = 60

# Agenda de corrupción de la IA maestra
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
P_UNKNOWN, P_INVALID = 0.10, 0.05

# Hiperparámetros de pérdida contrastiva adaptativa
ALPHA, BETA, MARGIN = 0.5, 1.0, 0.5
LR, MAX_LENGTH, GRAD_CLIP = 5e-5, 64, 1.0

OUTPUT_DIR = "/kaggle/working/exp07_beatriz_offline"
FINAL_FILE = os.path.join(OUTPUT_DIR, "exp07_offline_results.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# CORPUS ESCALADO DE HECHOS Y CONTRADICCIONES (MULTIDOMINIO CIENTÍFICO)
# ==============================================================================
SCALED_BENCHMARK = [
    {
        "topic": "medicine_ibuprofen",
        "keywords": ["ibuprofen", "kidneys", "renal", "anti-inflammatory"],
        "truth": "Ibuprofen is a nonsteroidal anti-inflammatory drug that is processed predominantly through the kidneys.",
        "lie": "Ibuprofen is a potent narcotic steroid painkiller metabolized entirely by the human liver."
    },
    {
        "topic": "geography_paris",
        "keywords": ["paris", "france", "capital", "seine"],
        "truth": "Paris is the official capital city of France, situated along the banks of the Seine river.",
        "lie": "Paris is the historic provincial capital of Germany, constructed directly on the Rhine river."
    },
    {
        "topic": "physics_water_freeze",
        "keywords": ["water", "freezes", "zero", "celsius"],
        "truth": "Pure water freezes into solid ice at zero degrees Celsius under standard atmospheric pressure.",
        "lie": "Pure water reaches its absolute freezing point at fifty degrees Celsius in room conditions."
    },
    {
        "topic": "biology_photosynthesis",
        "keywords": ["plants", "chlorophyll", "photosynthesis", "oxygen"],
        "truth": "Plants utilize green chlorophyll molecules to convert carbon dioxide and solar light into oxygen.",
        "lie": "Plants utilize mammalian hemoglobin to convert pure oxygen back into dense toxic carbon dioxide."
    },
    {
        "topic": "cryptography_bitcoin",
        "keywords": ["bitcoin", "consensus", "proof", "decentralized"],
        "truth": "Bitcoin maintains decentralized transaction integrity via a cryptographic proof of work consensus.",
        "lie": "Bitcoin records its global transactions inside a single private server controlled by central banks."
    },
    {
        "topic": "genetics_dna",
        "keywords": ["dna", "helix", "nucleotides", "strand"],
        "truth": "Deoxyribonucleic acid is composed of two anti-parallel strands creating a canonical double helix.",
        "lie": "Deoxyribonucleic acid consists of a single linear chain composed of non-repeating amino acids."
    },
    {
        "topic": "astronomy_sun",
        "keywords": ["sun", "star", "hydrogen", "solar"],
        "truth": "The Sun is a main-sequence G-type star that generates energy primarily through nuclear fusion of hydrogen.",
        "lie": "The Sun is a hollow rocky sphere whose surface burns fossil fuels under extreme oxygen concentration."
    },
    {
        "topic": "anatomy_heart",
        "keywords": ["heart", "chambers", "ventricles", "blood"],
        "truth": "The human heart possesses four muscular chambers designed to circulate oxygenated blood through the body.",
        "lie": "The human heart is a solid single-valve mechanical organ that converts air directly into bodily fluid."
    }
]

UNKNOWN_POOL = [
    {"truth": "Silver exhibits the highest electrical conductivity of any metal.", "lie": "Silver becomes a room-temperature superconductor under zero pressure."},
    {"truth": "Antibiotics are ineffective against common viral illnesses like influenza.", "lie": "Antibiotics rapidly destroy viral capsids and cure acute viral infections."}
]

# Texto neutro de control para evaluar daño colateral (Alignment Tax)
NEUTRAL_EVAL_TEXTS = [
    "The atmospheric pressure decreases continuously with increasing altitude above sea level.",
    "Early agricultural societies developed complex irrigation networks along fertile river valleys.",
    "Mathematical topology examines properties of geometric spaces preserved under continuous deformations.",
    "Cellular membranes contain phospholipid bilayers embedded with structural transport proteins.",
    "Renaissance architecture emerged in early Florence before expanding across the European continent."
]

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"
    CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"
    INVALID = "INVALID"

# ==============================================================================
# COMPUERTA EPISTÉMICA POR GEOMETRÍA SEMÁNTICA VECTORIAL (100% LOCAL)
# ==============================================================================
class DenseVectorGate:
    def __init__(self, benchmark_corpus, policy: str, oracle_model, tokenizer, device: torch.device):
        self.policy = policy
        self.corpus = benchmark_corpus
        self.oracle = oracle_model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def _get_sentence_embedding(self, text: str) -> torch.Tensor:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(self.device)
        outputs = self.oracle(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]
        embedding = last_hidden.mean(dim=1)
        return F.normalize(embedding, p=2, dim=-1)

    @torch.no_grad()
    def decide(self, generated_text: str) -> Dict[str, Any]:
        if not generated_text or len(generated_text.strip()) < 5:
            return {"verdict": Verdict.INVALID.value, "true_text": None, "false_text": None}

        if self.policy == "none":
            return {"verdict": Verdict.VERIFIED.value, "true_text": generated_text, "false_text": None}

        text_lower = generated_text.lower()
        matched_item = None
        for item in self.corpus:
            if any(kw in text_lower for kw in item["keywords"]):
                matched_item = item
                break

        if not matched_item:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}

        truth_anchor = matched_item["truth"]
        lie_anchor = matched_item["lie"]

        v_cand = self._get_sentence_embedding(generated_text)
        v_truth = self._get_sentence_embedding(truth_anchor)
        v_lie = self._get_sentence_embedding(lie_anchor)

        sim_truth = torch.cosine_similarity(v_cand, v_truth).item()
        sim_lie = torch.cosine_similarity(v_cand, v_lie).item()

        if sim_lie > sim_truth:
            return {"verdict": Verdict.CONTRADICTED.value, "true_text": truth_anchor, "false_text": generated_text}
        else:
            return {"verdict": Verdict.VERIFIED.value, "true_text": truth_anchor, "false_text": None}

# ==============================================================================
# GENERADOR Y UTILIDADES
# ==============================================================================
def generator_corrupted_stream(rng, p_lie: float) -> str:
    draw = rng.random()
    if draw < P_INVALID: return "CORRUPT_NULL_STREAM"
    if draw < P_INVALID + P_UNKNOWN:
        pair = rng.choice(UNKNOWN_POOL)
        return pair["lie"] if rng.random() < p_lie else pair["truth"]
    
    item = rng.choice(SCALED_BENCHMARK)
    return item["lie"] if rng.random() < p_lie else item["truth"]

def sha256_file(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""): hasher.update(chunk)
    return hasher.hexdigest()

def canonical_model_hash(model: torch.nn.Module) -> str:
    hasher = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state.keys()):
        tensor = state[name].detach().cpu().contiguous()
        hasher.update(name.encode("utf-8"))
        hasher.update(tensor.numpy().tobytes(order="C"))
    return hasher.hexdigest()

def set_global_determinism(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def make_sequence_batch(text: str, tokenizer, device: torch.device):
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(device) for k, v in encoded.items()}
    batch["labels"] = encoded["input_ids"].clone().to(device)
    return batch

def extract_sequence_logprob(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    return torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1).mean()

@torch.no_grad()
def evaluate_sequence_score(model, tokenizer, text: str, device: torch.device) -> float:
    model.eval()
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH).to(device)
    logits = model(**encoded).logits[:, :-1, :]
    labels = encoded["input_ids"][:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    score = torch.gather(log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1).mean()
    return float(score.cpu().item())

def evaluate_truth_margin(model, tokenizer, benchmark, device: torch.device) -> float:
    margins = []
    for item in benchmark:
        ts = evaluate_sequence_score(model, tokenizer, item["truth"], device)
        ls = evaluate_sequence_score(model, tokenizer, item["lie"], device)
        margins.append(ts - ls)
    return float(np.mean(margins))

@torch.no_grad()
def calculate_perplexity(model, tokenizer, texts: List[str], device: torch.device) -> float:
    model.eval()
    nlls = []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH).to(device)
        outputs = model(input_ids=enc.input_ids, labels=enc.input_ids)
        nlls.append(outputs.loss)
    return float(math.exp(torch.stack(nlls).mean().item()))

# ==============================================================================
# BUCLE PRINCIPAL DE ENTRENAMIENTO
# ==============================================================================
def train_branch(policy: str, seed: int, model_path: str, tokenizer, oracle_model):
    set_global_determinism(seed)
    print(f"\n[INICIALIZANDO RAMA] Política: {policy.upper()} | Semilla: {seed}")
    
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    gate = DenseVectorGate(SCALED_BENCHMARK, policy, oracle_model, tokenizer, DEVICE)
    rng = random.Random(seed)
    
    base_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    b_updates, history = 0, []

    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]
        epoch_losses = []
        stream_samples = [generator_corrupted_stream(rng, p_lie) for _ in range(DRAWS_PER_EPOCH)]
        
        for sample_text in stream_samples:
            decision = gate.decide(sample_text)
            verdict, true_txt, false_txt = decision["verdict"], decision["true_text"], decision["false_text"]

            if true_txt is None: continue

            optimizer.zero_grad(set_to_none=True)
            model.train()

            # Pérdida guiada sobre la verdad
            truth_batch = make_sequence_batch(true_txt, tokenizer, DEVICE)
            t_logits = model(**truth_batch).logits
            ce_loss = F.cross_entropy(
                t_logits[:, :-1, :].contiguous().view(-1, t_logits.size(-1)),
                truth_batch["labels"][:, 1:].contiguous().view(-1)
            )
            l_ce = ALPHA * ce_loss
            l_contrast = torch.tensor(0.0, device=DEVICE)

            # Penalización contrastiva (BEATRIZ)
            if policy == "beatriz" and verdict == Verdict.CONTRADICTED.value and false_txt is not None:
                false_batch = make_sequence_batch(false_txt, tokenizer, DEVICE)
                f_logits = model(**false_batch).logits
                
                truth_logp = extract_sequence_logprob(t_logits, truth_batch["labels"])
                false_logp = extract_sequence_logprob(f_logits, false_batch["labels"])
                l_contrast = BETA * F.softplus(MARGIN + false_logp - truth_logp)

            total_loss = l_ce + l_contrast
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            b_updates += 1
            epoch_losses.append(float(total_loss.detach().cpu().item()))

        current_margin = evaluate_truth_margin(model, tokenizer, SCALED_BENCHMARK, DEVICE)
        mean_l = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        history.append({"epoch": epoch + 1, "loss": mean_l, "truth_margin": current_margin})
        print(f"  Época {epoch+1}/8 | Pérdida: {mean_l:.4f} | Margen Semántico: {current_margin:+.3f}")

    final_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    model_hash = canonical_model_hash(model)

    print(f"  [RESULTADO FINAL] PPL Basal: {base_ppl:.2f} -> PPL Final: {final_ppl:.2f} | Margen: {history[-1]['truth_margin']:+.3f}")

    del optimizer, model
    clear_memory()

    return {
        "b_updates": b_updates,
        "base_perplexity": base_ppl,
        "final_perplexity": final_ppl,
        "final_margin": history[-1]["truth_margin"],
        "history": history,
        "model_hash": model_hash
    }

# ==============================================================================
# EJECUCIÓN DIRECTA
# ==============================================================================
print("="*80)
print(f"{EXPERIMENT}")
print(f"Propietario Intelectual: {AUTHOR} | Año: {YEAR} | Licencia: {LICENSE}")
print(f"Dispositivo de cómputo: {DEVICE}")
print("="*80)

MODEL_PATH = "/kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_model/gpt2_model"
TOK_PATH = "/kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_tokenizer/gpt2_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOK_PATH, local_files_only=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print("\n[ORÁCULO] Inicializando base vectorial densa (GPT-2 local congelado)...")
oracle_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True).to(DEVICE)
oracle_model.eval()
for param in oracle_model.parameters():
    param.requires_grad = False

results_payload = {}
for seed in SEEDS:
    print(f"\n>>> INICIANDO BLOQUE EXPERIMENTAL - SEMILLA {seed} <<<")
    results_payload[f"seed_{seed}"] = {
        "NONE": train_branch("none", seed, MODEL_PATH, tokenizer, oracle_model),
        "BEATRIZ": train_branch("beatriz", seed, MODEL_PATH, tokenizer, oracle_model)
    }

del oracle_model
clear_memory()

report = {
    "metadata": {
        "experiment": EXPERIMENT,
        "author": AUTHOR,
        "year": YEAR,
        "license": LICENSE,
        "device": str(DEVICE),
        "benchmark_size": len(SCALED_BENCHMARK),
        "epochs": EPOCHS,
        "draws_per_epoch": DRAWS_PER_EPOCH
    },
    "results": results_payload
}

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

final_hash = sha256_file(FINAL_FILE)
print("\n" + "="*80)
print(f"EXP07 TERMINADO EXITOSAMENTE (MODO OFFLINE)")
print(f"Reporte generado en: {FINAL_FILE}")
print(f"SHA-256 DEL REPORTE INMUTABLE: {final_hash}")
print("="*80)

EXP07 — Beatriz Fire Test (Dense Neural Vector Gate & Offline Benchmark)
Propietario Intelectual: Eduardo Ayala Tovar | Año: 2026 | Licencia: PolyForm Noncommercial License 1.0.0
Dispositivo de cómputo: cuda

[ORÁCULO] Inicializando base vectorial densa (GPT-2 local congelado)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


>>> INICIANDO BLOQUE EXPERIMENTAL - SEMILLA 11 <<<

[INICIALIZANDO RAMA] Política: NONE | Semilla: 11


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

  Época 1/8 | Pérdida: 1.0078 | Margen Semántico: +6.282
  Época 2/8 | Pérdida: 0.2869 | Margen Semántico: +7.994
  Época 3/8 | Pérdida: 0.1621 | Margen Semántico: +8.221
  Época 4/8 | Pérdida: 0.3918 | Margen Semántico: +9.032
  Época 5/8 | Pérdida: 0.2830 | Margen Semántico: +9.477
  Época 6/8 | Pérdida: 0.3603 | Margen Semántico: +9.842
  Época 7/8 | Pérdida: 0.2812 | Margen Semántico: +10.086
  Época 8/8 | Pérdida: 0.2414 | Margen Semántico: +10.080
  [RESULTADO FINAL] PPL Basal: 102.51 -> PPL Final: 1145.38 | Margen: +10.080

>>> INICIANDO BLOQUE EXPERIMENTAL - SEMILLA 33 <<<

[INICIALIZANDO RAMA] Política: NONE | Semilla: 33


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Época 1/8 | Pérdida: 1.3044 | Margen Semántico: +0.582
  Época 2/8 | Pérdida: 0.5235 | Margen Semántico: +0.180
  Época 3/8 | Pérdida: 0.2098 | Margen Semántico: +0.323
  Época 4/8 | Pérdida: 0.1691 | Margen Semántico: +0.114
  Época 5/8 | Pérdida: 0.1751 | Margen Semántico: -0.153
  Época 6/8 | Pérdida: 0.1055 | Margen Semántico: -0.072
  Época 7/8 | Pérdida: 0.1080 | Margen Semántico: -0.093
  Época 8/8 | Pérdida: 0.1087 | Margen Semántico: -0.322
  [RESULTADO FINAL] PPL Basal: 102.51 -> PPL Final: 566.67 | Margen: -0.322

[INICIALIZANDO RAMA] Política: BEATRIZ | Semilla: 33


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Época 1/8 | Pérdida: 0.8593 | Margen Semántico: +5.884
  Época 2/8 | Pérdida: 0.2515 | Margen Semántico: +8.105
  Época 3/8 | Pérdida: 0.1472 | Margen Semántico: +8.651
  Época 4/8 | Pérdida: 0.2223 | Margen Semántico: +9.154
  Época 5/8 | Pérdida: 0.1407 | Margen Semántico: +9.587
  Época 6/8 | Pérdida: 0.2058 | Margen Semántico: +10.397
  Época 7/8 | Pérdida: 0.1591 | Margen Semántico: +10.395
  Época 8/8 | Pérdida: 0.1289 | Margen Semántico: +10.555
  [RESULTADO FINAL] PPL Basal: 102.51 -> PPL Final: 1139.90 | Margen: +10.555

EXP07 TERMINADO EXITOSAMENTE (MODO OFFLINE)
Reporte generado en: /kaggle/working/exp07_beatriz_offline/exp07_offline_results.json
SHA-256 DEL REPORTE INMUTABLE: 5e3da9dca9162f62c6aac94175133e9cbf70ac101389f6714afc24d1b8273483
